# MetabTravLR — submit SpaceTravLR runs to SLURM

One cell per dataset. Running a cell submits SLURM job(s) and returns immediately with the
job id(s). Nothing heavy happens in this notebook.

What the job does (`run_spacetravlr.py`):

| stage | needs | writes |
|---|---|---|
| `setup` | **CPU + lots of RAM** | `spacetravlr_output/input_data/` (processed adata, CellOracle links, NicheNet links) |
| `fit` | **GPU** | `spacetravlr_output/betadata/<gene>_betadata.parquet` |
| `artifacts` | CPU | `easy_download/metabtravlr_outputs/<tier>/{gene_pairs.csv, histograms.csv, histograms.png}` and `spacetravlr_adata.h5ad` |

Setup never touches the GPU (imputation is `magic`, the GRN is sklearn ridge) but it copies
the full AnnData several times — `process_adata_` and CellOracle each do — so at Xenium
scale it needs far more RAM than a GPU allocation provides. Hence **`submit_split()`**:
setup on a CPU big-mem node, then the GPU job chained behind it with
`--dependency=afterok`. Small datasets can still use plain `submit()` for one combined job.

**Logs** go to `{METAB_DATA_DIR}/spacetravlr_logs/<DATASET>/<stages>_<timestamp>.log` — a
sibling of `harreman_logs/`, deliberately *outside* `spacetravlr_output/`. SLURM opens
that file before the job body runs, so it cannot live in the directory the job is about
to create.

**Reruns are cheap and safe.** Setup is skipped when it's already complete, and `fit` skips
any gene that already has a betadata parquet — so re-submitting after a timeout resumes
rather than starting over.

Per-dataset settings (cell-type column, tiers, target genes, SLURM resources) live in
`dataset_configs.py`. Target genes are shared across datasets by default
(`metab_travlr_config.FOCUS_GENES`); metabolite pairs always come from each dataset's own
harreman `metabolite_selection.yaml`.

In [ ]:
import sys
from pathlib import Path
# Make the repo root importable regardless of CWD or machine: walk up from the
# working dir until we hit a repo marker, then put that dir on sys.path.
_start = Path.cwd()
_root = next((p for p in (_start, *_start.parents)
              if (p / ".git").exists() or (p / "setup.py").exists()), _start)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from metab_processing.SpaceTravLR.submit_spacetravlr import submit, submit_split
from metab_processing.SpaceTravLR.dataset_configs import DATASETS, get_config

print('datasets:', sorted(DATASETS))
print('target genes:', get_config(sorted(DATASETS)[0])['focus_genes'])

## Primary_Dermal_Melanoma

In [ ]:
# Fits in one GPU job. Use submit_split(...) instead if setup ever OOMs.
submit('Primary_Dermal_Melanoma', overwrite=True, time_hours=24)

## Human_Lung

In [ ]:
# 278k cells x 5k genes: setup peaks ~50 GB and OOMs an 8-core A40 allocation, so it
# gets its own CPU big-mem job and the GPU job waits on it (--dependency=afterok).
submit_split('Human_Lung', overwrite=True, run={'time_hours': 24})

## Variations

**`submit_split(dataset)`** — two chained jobs: `setup` on a CPU big-mem node, then
`fit`+`artifacts` on the GPU, the second waiting on `--dependency=afterok` of the first.
Both queue immediately, so the GPU job builds priority while setup runs, and it never
starts if setup fails. `overwrite=` goes to the setup job, `clear_betadata=` to the
training job; `setup={...}` / `run={...}` override SLURM keys per job.

**`submit(dataset)`** — one job for all three stages. Fine when setup fits in the GPU
allocation's memory.

Both take:

- `stages=['setup'|'fit'|'artifacts']` — run only part of the pipeline. A **setup-only**
  submission automatically uses the CPU big-mem profile (`setup_slurm` in
  `dataset_configs.py`); everything else uses the GPU profile. The printed `profile:` line
  says which.
- `overwrite=True` — delete `input_data/` and redo setup. Needs the `setup` stage (it is
  rejected rather than silently ignored otherwise). **Trained betadata is kept**, so those
  betas came from the *previous* preprocessing — the job log says so.
- `clear_betadata=True` — delete `betadata/`, forcing every gene to retrain. Independent of
  `overwrite`; combine the two for a genuinely clean slate.
- `dry_run=True` — print the sbatch settings and command without submitting.
- any SLURM key as a keyword (`time_hours`, `partition`, `qos`, `gres`, `cpus_per_task`,
  `account`, `python_path`) to override the config for this submission only.

Two jobs may `fit` the same dataset concurrently (the gene queue is lock-based), but **not
`setup`** — a second setup on a dataset already being set up refuses, via a `.setup.lock`
in `spacetravlr_output/`. If a setup job was killed, delete that file before resubmitting.

In [ ]:
# See exactly what would be submitted, without submitting.
submit('Human_Lung', dry_run=True)

In [ ]:
# Re-do just the read-out after a finished training run (CPU-only, minutes not hours).
# submit('Primary_Dermal_Melanoma', stages=['artifacts'], gres=None, time_hours=2)

# Redo setup from scratch and retrain every gene.
# submit_split('Human_Lung', overwrite=True, clear_betadata=True)

# If the TRAINING job OOMs too: memory scales with cores, so ask for more of them.
# fit opens _adata.h5ad with X + raw_count + imputed_count (~17 GB for Human_Lung)
# before it starts.
# submit_split('Human_Lung', run={'cpus_per_task': 16, 'time_hours': 36})

# Spawn a second worker on a dataset already training -- the gene queue is lock-based,
# so extra workers just pick up untrained genes.
# submit('Human_Lung', stages=['fit'])

## Monitor

In [12]:
!squeue -u $USER

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)
          36157087 ood-inter OOD_VSCo fosteran  R    2:19:22      1 n0002.savio4
          36169480 savio3_gp MetabTra fosteran  R       0:28      1 n0264.savio3
          36162713 savio3_gp MetabTra fosteran  R      49:51      1 n0210.savio3


In [16]:
# Tail the newest log for a dataset.
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths

# DATASET = 'Primary_Dermal_Melanoma'
DATASET = 'Human_Lung'
logs = sorted(dataset_paths(DATASET)['log_dir'].glob('*.log'))
print(logs[-1] if logs else 'no logs yet')
if logs:
    print(''.join(logs[-1].read_text().splitlines(keepends=True)[-40:]))

/global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/spacetravlr_logs/Human_Lung/all_20260731_131933.log
[13:20:17] === Human_Lung | stages=['setup', 'fit', 'artifacts'] | overwrite=True ===
[13:20:17] outdir: /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/spacetravlr_output
[13:20:17] XDG_CACHE_HOME=/tmp/spacetravlr_cache_36169480
[13:20:17] overwrite: removing /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/spacetravlr_output/input_data
[13:20:17] reading /global/scratch/fsa/fc_wagnerlabfca/fosterangus/MetabTravLR/Data/Xenium/Human_Lung/adata.h5ad
[13:20:21] adata: 278328 cells x 5001 genes
[13:20:21] setup_ (run_commot=False) ...



In [ ]:
# What has finished training so far.
for dataset in sorted(DATASETS):
    paths = dataset_paths(dataset)
    done = sorted(p.name[:-len('_betadata.parquet')]
                  for p in paths['betadata'].glob('*_betadata.parquet')) \
        if paths['betadata'].exists() else []
    print(f"{dataset:28s} setup={paths['input_data'].is_dir()}  "
          f"beta_adata={paths['beta_adata'].exists()}  genes={done}")

## Publish

Copy each dataset's `easy_download/` (now including `metabtravlr_outputs/`) into the
aggregate `Results/` tree — the same step `quick_start_metab.ipynb` and `run_full_harr.ipynb`
end with. Deliberately **not** part of the job: it touches every dataset, not just the one
that ran, so it's a manual step once the runs you care about have finished.

In [ ]:
sys.path.insert(0, str(_root / 'metab_processing'))
from Harreman.copy_easy_download import save_easy_downloads

save_easy_downloads()